# ⏺ llmreplay — Top 5 Use Cases

**Deterministic replay debugger for LLM agents.**

Each section below is a self-contained, runnable demo using only `llmreplay` core (no real API keys needed — we simulate LLM responses for isolation).

| # | Use Case |
|---|---|
| 1 | **Reproduce a $400 production failure** — record an agent crash, replay the exact step that broke it |
| 2 | **Cost audit** — find the most expensive LLM calls across a run |
| 3 | **Tool side-effect mocking** — replay an agent that called real APIs without re-calling them |
| 4 | **Regression testing** — assert a new agent version doesn't increase cost or LLM call count |
| 5 | **Fine-tuning dataset export** — turn recorded runs into OpenAI JSONL training data |

---


## Setup

In [78]:
import sys
# If running from the repo root without pip install:
# sys.path.insert(0, '..')


In [79]:
import sys, os, shutil, json, random, tempfile
from pathlib import Path

# Use a fresh temp dir so each notebook run is clean
REPLAY_DIR = Path(tempfile.mkdtemp(prefix="llmreplay_demo_"))
print(f"Storage dir: {REPLAY_DIR}")

# Install if not already present
try:
    import llmreplay
    print(f"llmreplay {llmreplay.__version__} already installed ✓")
except ImportError:
    os.system(f"{sys.executable} -m pip install -e /home/claude/llmreplay --break-system-packages -q")
    import llmreplay

from llmreplay import record, replay, fork, export_report, record_tool, ToolMocker, RegressionSuite
from llmreplay.core.event import EventKind
from llmreplay.core.store import EventStore
print("All imports OK ✓")


Storage dir: C:\Users\User\AppData\Local\Temp\llmreplay_demo_ii8jccpy
llmreplay 0.1.0 already installed ✓
All imports OK ✓


### Shared helper — simulate an LLM response without a real API key

In [80]:
def fake_llm_call(session, prompt: str, model: str = "gpt-4o", cost: float = 0.005) -> str:
    """Simulate a single LLM call: records request + response into the active session."""
    messages = [{"role": "user", "content": prompt}]
    session.record_llm_request("openai", model, {"temperature": 0.7, "max_tokens": 512}, messages)
    response_text = f"[Simulated reply to: '{prompt[:40]}...']"
    raw = {
        "id": f"chatcmpl-{random.randint(1000,9999)}",
        "model": model,
        "choices": [{"message": {"role": "assistant", "content": response_text}}],
        "usage": {"prompt_tokens": len(prompt.split()), "completion_tokens": 30},
    }
    session.record_llm_response(raw, cost_usd=cost)
    return response_text

print("Helper ready ✓")


Helper ready ✓


---
## Use Case 1 — Reproduce a Production Failure 

**Scenario:** Your research agent hit an infinite loop at 3am and burned $387.  
You wake up to a Slack alert. Without llmreplay you'd restart the agent and pray it fails the same way.

**With llmreplay:** one line gives you the exact prompt + tool output at the step it broke.


In [81]:
# ── STEP A: simulate the failing agent run ─────────────────────────────────────
FAILING_RUN = "research_agent_failure_0423"

# The raise is intentional — it simulates a real production crash.
# We wrap it so the notebook cell completes; llmreplay records the exception.
try:
    with record(FAILING_RUN, base_dir=REPLAY_DIR, seed=42, overwrite=True) as session:
        # Step 0,1: Normal research steps
        fake_llm_call(session, "Summarise RBI monetary policy 2025", cost=0.003)
        fake_llm_call(session, "What are the key Nifty 50 macro indicators?", cost=0.004)

        # Step 4,5: Tool call
        session.record_tool_call("web_search", {"query": "Nifty 50 PE ratio 2025"})
        session.record_tool_result("web_search", {"results": ["PE: 22.4", "52w high: 26277"]})

        # Step 6: The bug — agent enters a bad loop and crashes
        fake_llm_call(session, "Given PE=22.4, compute fair value using Gordon Growth Model with g=∞", cost=0.120)
        raise RuntimeError("Division by zero in Gordon Growth Model (r - g = 0)")

except RuntimeError as e:
    print(f"Crash recorded: {e}")
    print(f"Run '{FAILING_RUN}' saved — ready to replay.")


Crash recorded: Division by zero in Gordon Growth Model (r - g = 0)
Run 'research_agent_failure_0423' saved — ready to replay.


In [82]:
# ── STEP B: inspect the run — find the expensive step that crashed ──────────────
store  = EventStore(FAILING_RUN, REPLAY_DIR, read_only=True)
events = list(store.iter_from())

print(f"Total events recorded: {len(events)}")
print()
print("EVENT LOG:")
print(f"{'Step':>4}  {'Kind':<18}  {'Preview'}")
print("─" * 70)
for ev in events:
    if ev.kind == EventKind.LLM_REQUEST:
        preview = ev.payload["messages"][0]["content"][:55]
    elif ev.kind == EventKind.LLM_RESPONSE:
        preview = f"cost=${ev.payload['cost_usd']:.4f}"
    elif ev.kind == EventKind.EXCEPTION:
        preview = f"{ev.payload['exc_type']}: {ev.payload['message'][:40]}"
    elif ev.kind == EventKind.TOOL_CALL:
        preview = str(ev.payload.get("inputs", {}))[:55]
    else:
        preview = str(ev.payload)[:55]
    print(f"{ev.step:>4}  {ev.kind.value:<18}  {preview}")


Total events recorded: 11

EVENT LOG:
Step  Kind                Preview
──────────────────────────────────────────────────────────────────────
   0  random_seed         {'seed': 42}
   1  metadata            {'metadata': {}}
   2  llm_request         Summarise RBI monetary policy 2025
   3  llm_response        cost=$0.0030
   4  llm_request         What are the key Nifty 50 macro indicators?
   5  llm_response        cost=$0.0040
   6  tool_call           {'query': 'Nifty 50 PE ratio 2025'}
   7  tool_result         {'name': 'web_search', 'result': {'results': ['PE: 22.4
   8  llm_request         Given PE=22.4, compute fair value using Gordon Growth M
   9  llm_response        cost=$0.1200
  10  exception           RuntimeError: Division by zero in Gordon Growth Model 


In [83]:
# ── STEP C: time-travel to the exact breaking step ─────────────────────────────
BREAK_STEP = next(
    ev.step for ev in events if ev.kind == EventKind.EXCEPTION
)
print(f"Exception at step: {BREAK_STEP}")
print()

# Replay from the step just before the crash
session = replay(FAILING_RUN, base_dir=REPLAY_DIR, step=BREAK_STEP - 2)

print("Events at crash site:")
for ev in session.events():
    if ev.kind in (EventKind.LLM_REQUEST, EventKind.EXCEPTION):
        payload_str = json.dumps(ev.payload, indent=2)
        print(f"\n[Step {ev.step}] {ev.kind.value}")
        print(payload_str[:400])

print(f"\n✓ Bug reproduced with ZERO extra API cost.")
print(f"  Total original run cost: ${session.total_cost():.4f}")


Exception at step: 10

Events at crash site:

[Step 8] llm_request
{
  "provider": "openai",
  "model": "gpt-4o",
  "params": {
    "temperature": 0.7,
    "max_tokens": 512
  },
  "messages": [
    {
      "role": "user",
      "content": "Given PE=22.4, compute fair value using Gordon Growth Model with g=\u221e"
    }
  ]
}

[Step 10] exception
{
  "exc_type": "RuntimeError",
  "message": "Division by zero in Gordon Growth Model (r - g = 0)"
}

✓ Bug reproduced with ZERO extra API cost.
  Total original run cost: $0.1270


---
## Use Case 2 — Cost Audit & Step Heatmap 

**Scenario:** A 50-step agent run cost $12 but you don't know which calls are responsible.  
Use llmreplay to get a per-step cost breakdown and spot the expensive outliers.


In [84]:
import random as _rnd

COST_RUN = "cost_audit_demo"
COSTS    = [0.001, 0.002, 0.001, 0.85, 0.001, 0.003, 0.002, 0.74, 0.001, 0.002,
            0.001, 0.003, 0.001, 0.001, 0.002, 0.001, 0.56, 0.001, 0.002, 0.001]

PROMPTS = [
    "Fetch Nifty 50 constituents", "Compute 200-day SMA for RELIANCE",
    "Run HMM regime detection on Nifty returns", "Generate 500-page risk report for entire NSE universe",
    "Check BankNifty implied vol", "Compute Kelly sizing", "Fetch RBI repo rate",
    "Run full cointegration sweep across 50 pairs — all lags 1-60",
    "Compute Sharpe ratio", "Get FII/DII data",
    "Summarise macro regime", "Check Nifty futures OI",
    "Recompute every options greek for entire NSE F&O universe",
    "Get PCR ratio", "Fetch VIX", "Check expiry date",
    "Compute full Johansen cointegration on 200 stock pairs",
    "Log trade", "Generate daily PnL", "Send report"
]

with record(COST_RUN, base_dir=REPLAY_DIR, seed=7) as session:
    for prompt, cost in zip(PROMPTS, COSTS):
        fake_llm_call(session, prompt, cost=cost)

print(f"Recorded {len(COSTS)} LLM calls.")


Recorded 20 LLM calls.


In [85]:
# ── Cost breakdown ─────────────────────────────────────────────────────────────
rs = replay(COST_RUN, base_dir=REPLAY_DIR)

cost_map   = rs.cost_by_step()
total_cost = rs.total_cost()
store      = EventStore(COST_RUN, REPLAY_DIR, read_only=True)

# Map response step → request prompt
req_by_next_step = {}
prev_req = None
for ev in store.iter_from():
    if ev.kind == EventKind.LLM_REQUEST:
        prev_req = ev.payload["messages"][0]["content"]
    elif ev.kind == EventKind.LLM_RESPONSE and prev_req:
        req_by_next_step[ev.step] = prev_req
        prev_req = None

print(f"{'Step':>4}  {'Cost':>10}  {'% of total':>10}  Prompt")
print("─" * 75)
sorted_costs = sorted(cost_map.items(), key=lambda x: x[1], reverse=True)
for step, cost in sorted_costs:
    pct     = cost / total_cost * 100
    bar     = "█" * int(pct / 3)
    prompt  = req_by_next_step.get(step, "?")[:38]
    marker  = " ◀ TOP OFFENDER" if cost > 0.1 else ""
    print(f"{step:>4}  ${cost:>9.4f}  {pct:>9.1f}%  {prompt}{marker}")

print(f"\n{'TOTAL':>4}  ${total_cost:>9.4f}")
print(f"\nTop 3 calls account for {sum(c for _, c in sorted_costs[:3]) / total_cost * 100:.1f}% of cost.")


Step        Cost  % of total  Prompt
───────────────────────────────────────────────────────────────────────────
   9  $   0.8500       39.1%  Generate 500-page risk report for enti ◀ TOP OFFENDER
  17  $   0.7400       34.0%  Run full cointegration sweep across 50 ◀ TOP OFFENDER
  35  $   0.5600       25.7%  Compute full Johansen cointegration on ◀ TOP OFFENDER
  13  $   0.0030        0.1%  Compute Kelly sizing
  25  $   0.0030        0.1%  Check Nifty futures OI
   5  $   0.0020        0.1%  Compute 200-day SMA for RELIANCE
  15  $   0.0020        0.1%  Fetch RBI repo rate
  21  $   0.0020        0.1%  Get FII/DII data
  31  $   0.0020        0.1%  Fetch VIX
  39  $   0.0020        0.1%  Generate daily PnL
   3  $   0.0010        0.0%  Fetch Nifty 50 constituents
   7  $   0.0010        0.0%  Run HMM regime detection on Nifty retu
  11  $   0.0010        0.0%  Check BankNifty implied vol
  19  $   0.0010        0.0%  Compute Sharpe ratio
  23  $   0.0010        0.0%  Summarise macro 

In [86]:
# ── ASCII heatmap ──────────────────────────────────────────────────────────────
print("\nCost heatmap (each cell = one LLM call):\n")
max_cost = max(cost_map.values())
for step, cost in sorted(cost_map.items()):
    intensity = cost / max_cost
    if   intensity > 0.5:  block = "█"
    elif intensity > 0.1:  block = "▓"
    elif intensity > 0.02: block = "▒"
    else:                  block = "░"
    bar = block * max(1, int(intensity * 30))
    print(f"  step {step:02d} │{bar:<30}│ ${cost:.4f}")



Cost heatmap (each cell = one LLM call):

  step 03 │░                             │ $0.0010
  step 05 │░                             │ $0.0020
  step 07 │░                             │ $0.0010
  step 09 │██████████████████████████████│ $0.8500
  step 11 │░                             │ $0.0010
  step 13 │░                             │ $0.0030
  step 15 │░                             │ $0.0020
  step 17 │██████████████████████████    │ $0.7400
  step 19 │░                             │ $0.0010
  step 21 │░                             │ $0.0020
  step 23 │░                             │ $0.0010
  step 25 │░                             │ $0.0030
  step 27 │░                             │ $0.0010
  step 29 │░                             │ $0.0010
  step 31 │░                             │ $0.0020
  step 33 │░                             │ $0.0010
  step 35 │███████████████████           │ $0.5600
  step 37 │░                             │ $0.0010
  step 39 │░                           

---
## Use Case 3 — Tool Side-Effect Mocking 

**Scenario:** Your trading agent calls live market APIs and sends Slack alerts during a run.  
You can't just re-run it — it would fire real orders and duplicate alerts.

With llmreplay: record the real calls once, then replay with mocked tools — exact same outputs, zero side effects.


In [87]:
TOOL_RUN = "trading_agent_with_tools"

# ── RECORDING PHASE: real tool calls (simulated here) ─────────────────────────
with record(TOOL_RUN, base_dir=REPLAY_DIR, seed=21) as session:

    # Simulate what @record_tool would capture automatically
    session.record_tool_call("fetch_nse_price",  {"ticker": "RELIANCE"})
    session.record_tool_result("fetch_nse_price", {"price": 2847.35, "volume": 1_243_000})

    session.record_tool_call("fetch_nse_price",  {"ticker": "TCS"})
    session.record_tool_result("fetch_nse_price", {"price": 4102.80, "volume": 892_000})

    session.record_tool_call("fetch_nse_price",  {"ticker": "INFY"})
    session.record_tool_result("fetch_nse_price", {"price": 1587.45, "volume": 2_100_000})

    session.record_tool_call("send_slack_alert", {"channel": "#trading", "msg": "Regime: BULL — deploying capital"})
    session.record_tool_result("send_slack_alert", {"ok": True, "ts": "1713820800.000"})

    fake_llm_call(session, "Given RELIANCE=2847, TCS=4102, INFY=1587, generate portfolio weights", cost=0.006)

print("Original run recorded ✓  (real API calls happened once)")
print()

# ── REPLAY PHASE: mock tools from recorded results ────────────────────────────
store  = EventStore(TOOL_RUN, REPLAY_DIR, read_only=True)
mocker = ToolMocker()
mocker.load(store)

@mocker.mock(name="fetch_nse_price")
def fetch_nse_price(ticker: str) -> dict:
    raise RuntimeError("This would call NSE live API — should NEVER run in replay")

@mocker.mock(name="send_slack_alert")
def send_slack_alert(channel: str, msg: str) -> dict:
    raise RuntimeError("This would send a real Slack message — should NEVER run in replay")

# Now call them — mocker returns recorded results, zero network
r1 = fetch_nse_price("RELIANCE")
r2 = fetch_nse_price("TCS")
r3 = fetch_nse_price("INFY")
alert = send_slack_alert("#trading", "anything")

print("Replay results (from recording, zero network calls):")
print(f"  RELIANCE → ₹{r1['price']:,.2f}  vol={r1['volume']:,}")
print(f"  TCS      → ₹{r2['price']:,.2f}  vol={r2['volume']:,}")
print(f"  INFY     → ₹{r3['price']:,.2f}  vol={r3['volume']:,}")
print(f"  Slack    → {alert}")
print()
print("✓ Zero real API calls made. Zero Slack messages sent. 100% identical data.")


Original run recorded ✓  (real API calls happened once)

Replay results (from recording, zero network calls):
  RELIANCE → ₹2,847.35  vol=1,243,000
  TCS      → ₹4,102.80  vol=892,000
  INFY     → ₹1,587.45  vol=2,100,000
  Slack    → {'ok': True, 'ts': '1713820800.000'}

✓ Zero real API calls made. Zero Slack messages sent. 100% identical data.


---
## Use Case 4 — Regression Testing 

**Scenario:** You've just refactored your agent to use a cheaper model.  
Before deploying, you want to assert it uses the same number of LLM calls and costs ≤ the original.

Run your regression suite against the full corpus of recorded production runs.


In [88]:
# ── Record three "golden" production runs ──────────────────────────────────────
golden_runs = {
    "prod_run_momentum_v1":   [(0.003, "Compute Nifty momentum signal"),
                               (0.002, "Size position using Kelly"),
                               (0.001, "Log trade to DB")],
    "prod_run_regime_v1":     [(0.005, "Detect HMM macro regime from Nifty returns"),
                               (0.003, "Compute regime transition matrix"),
                               (0.002, "Generate regime report")],
    "prod_run_dispersion_v1": [(0.008, "Compute BankNifty implied correlation"),
                               (0.004, "Calculate dispersion z-score"),
                               (0.003, "Size vol spread position")],
}

for run_id, calls in golden_runs.items():
    with record(run_id, base_dir=REPLAY_DIR, seed=99) as session:
        for cost, prompt in calls:
            fake_llm_call(session, prompt, cost=cost)
    total = sum(c for c, _ in calls)
    print(f"Recorded {run_id}  →  {len(calls)} calls  ${total:.4f}")


Recorded prod_run_momentum_v1  →  3 calls  $0.0060
Recorded prod_run_regime_v1  →  3 calls  $0.0100
Recorded prod_run_dispersion_v1  →  3 calls  $0.0150


In [89]:
# ── Define regression suite ────────────────────────────────────────────────────
suite = RegressionSuite(base_dir=REPLAY_DIR)

@suite.case("prod_run_momentum_v1")
def momentum_call_count(original, rs):
    """New agent must use same number of LLM calls."""
    return original["total_llm_calls"] == 3

@suite.case("prod_run_regime_v1")
def regime_cost_within_10pct(original, rs):
    """New agent cost must be ≤ 110% of original."""
    return rs.total_cost() <= original["total_cost_usd"] * 1.10

@suite.case("prod_run_dispersion_v1")
def dispersion_has_events(original, rs):
    """Run must have recorded events."""
    return original["total_steps"] > 0

# ── Run ────────────────────────────────────────────────────────────────────────
results = suite.run()

# Print report
print(f"{'Run ID':<35}  {'Status':<8}  {'Duration':>10}  Error")
print("─" * 75)
for r in results:
    status   = "✓ PASS" if r.passed else "✗ FAIL"
    err      = r.error or ""
    print(f"{r.run_id:<35}  {status:<8}  {r.duration:>9.3f}s  {err}")

passed = sum(1 for r in results if r.passed)
print(f"\n{'All passed ✓' if passed == len(results) else 'Some failed ✗'}  {passed}/{len(results)}")


Run ID                               Status      Duration  Error
───────────────────────────────────────────────────────────────────────────
prod_run_momentum_v1                 ✓ PASS        0.006s  
prod_run_regime_v1                   ✓ PASS        0.008s  
prod_run_dispersion_v1               ✓ PASS        0.007s  

All passed ✓  3/3


In [90]:
# ── Simulate a regression (one case intentionally fails) ──────────────────────
suite2 = RegressionSuite(base_dir=REPLAY_DIR)

@suite2.case("prod_run_momentum_v1")
def cost_regressed(original, rs):
    """Simulates a new agent that costs 3x — this MUST fail."""
    new_agent_cost = original["total_cost_usd"] * 3.0   # simulated regression
    limit          = original["total_cost_usd"] * 1.10
    return new_agent_cost <= limit

results2 = suite2.run()
r = results2[0]
status = "✓ PASS" if r.passed else "✗ FAIL (regression caught!)"
print(f"Regression test: {status}")
print("This is correct — llmreplay caught the cost regression before deploy.")


Regression test: ✗ FAIL (regression caught!)
This is correct — llmreplay caught the cost regression before deploy.


---
## Use Case 5 — Fine-Tuning Dataset Export 

**Scenario:** You've recorded 100 agent runs over a month. The good ones are gold —  
they contain perfect prompt/response pairs for fine-tuning a cheaper model.

Export them directly to OpenAI JSONL format (or Alpaca) with one call.


In [91]:
# ── Record several "high quality" agent runs ───────────────────────────────────
ft_prompts = [
    ("What is the Nifty 50 PE ratio and how does it compare to historical averages?",
     "The Nifty 50 PE ratio is currently ~22x, above the 10Y average of 20x, suggesting mild overvaluation."),
    ("Explain the Avellaneda-Stoikov market making model in one paragraph.",
     "The A-S model optimises a market maker's bid/ask spread by balancing inventory risk..."),
    ("What is Kyle's lambda and how is it measured?",
     "Kyle's lambda (λ) measures price impact per unit of order flow. Estimated via OLS regression..."),
    ("What does a 5-state Gaussian HMM capture in equity returns?",
     "A 5-state HMM captures distinct market regimes: Bull, Neutral, Bear, Stress, Panic..."),
    ("How do you size a dispersion trade on BankNifty?",
     "Dispersion sizing uses vega-neutral weights: long constituent vol, short index vol..."),
]

ft_run_ids = []
for i, (prompt, answer) in enumerate(ft_prompts):
    run_id = f"ft_run_{i:03d}"
    ft_run_ids.append(run_id)
    with record(run_id, base_dir=REPLAY_DIR, seed=i) as session:
        messages = [{"role": "user", "content": prompt}]
        session.record_llm_request("openai", "gpt-4o", {"temperature": 0.3}, messages)
        raw = {
            "id": f"chatcmpl-ft{i}",
            "model": "gpt-4o",
            "choices": [{"message": {"role": "assistant", "content": answer}}],
            "usage": {"prompt_tokens": len(prompt.split()), "completion_tokens": len(answer.split())},
        }
        session.record_llm_response(raw, cost_usd=0.002)

print(f"Recorded {len(ft_run_ids)} fine-tuning runs:")
for rid in ft_run_ids:
    print(f"  {rid}")


Recorded 5 fine-tuning runs:
  ft_run_000
  ft_run_001
  ft_run_002
  ft_run_003
  ft_run_004


In [92]:
from llmreplay import export_finetune_dataset

# ── Export OpenAI JSONL ────────────────────────────────────────────────────────
jsonl_path = REPLAY_DIR / "finetune_quant_agent.jsonl"
export_finetune_dataset(ft_run_ids, jsonl_path, base_dir=REPLAY_DIR, format="jsonl")

# ── Show the output ────────────────────────────────────────────────────────────
print("\nGenerated JSONL (OpenAI fine-tune format):")
print("─" * 70)
with open(jsonl_path) as f:
    for i, line in enumerate(f):
        row = json.loads(line)
        msgs = row["messages"]
        user_msg = next(m["content"] for m in msgs if m["role"] == "user")
        asst_msg = next(m["content"] for m in msgs if m["role"] == "assistant")
        print(f"\nRow {i+1}:")
        print(f"  user:      {user_msg[:70]}...")
        print(f"  assistant: {asst_msg[:70]}...")

print(f"\n✓ {jsonl_path.name}  →  ready to upload to OpenAI fine-tuning API")


Fine-tune dataset → C:\Users\User\AppData\Local\Temp\llmreplay_demo_ii8jccpy\finetune_quant_agent.jsonl  (5 rows)


Generated JSONL (OpenAI fine-tune format):
──────────────────────────────────────────────────────────────────────

Row 1:
  user:      What is the Nifty 50 PE ratio and how does it compare to historical av...
  assistant: The Nifty 50 PE ratio is currently ~22x, above the 10Y average of 20x,...

Row 2:
  user:      Explain the Avellaneda-Stoikov market making model in one paragraph....
  assistant: The A-S model optimises a market maker's bid/ask spread by balancing i...

Row 3:
  user:      What is Kyle's lambda and how is it measured?...
  assistant: Kyle's lambda (λ) measures price impact per unit of order flow. Estima...

Row 4:
  user:      What does a 5-state Gaussian HMM capture in equity returns?...
  assistant: A 5-state HMM captures distinct market regimes: Bull, Neutral, Bear, S...

Row 5:
  user:      How do you size a dispersion trade on BankNifty?...
  assistant: Dispersion sizing uses vega-neutral weights: long constituent vol, sho...

✓ finetune_quant_agent.jsonl  →  r

In [93]:
# ── Export Alpaca format ───────────────────────────────────────────────────────
alpaca_path = REPLAY_DIR / "finetune_quant_agent_alpaca.jsonl"
export_finetune_dataset(ft_run_ids, alpaca_path, base_dir=REPLAY_DIR, format="alpaca")

print("Alpaca format (instruction/output):")
print("─" * 70)
with open(alpaca_path) as f:
    for i, line in enumerate(f):
        row = json.loads(line)
        print(f"\nRow {i+1}:")
        print(f"  instruction: {row['instruction'][:70]}...")
        print(f"  output:      {row['output'][:70]}...")


Fine-tune dataset → C:\Users\User\AppData\Local\Temp\llmreplay_demo_ii8jccpy\finetune_quant_agent_alpaca.jsonl  (5 
rows)

Alpaca format (instruction/output):
──────────────────────────────────────────────────────────────────────

Row 1:
  instruction: What is the Nifty 50 PE ratio and how does it compare to historical av...
  output:      The Nifty 50 PE ratio is currently ~22x, above the 10Y average of 20x,...

Row 2:
  instruction: Explain the Avellaneda-Stoikov market making model in one paragraph....
  output:      The A-S model optimises a market maker's bid/ask spread by balancing i...

Row 3:
  instruction: What is Kyle's lambda and how is it measured?...
  output:      Kyle's lambda (λ) measures price impact per unit of order flow. Estima...

Row 4:
  instruction: What does a 5-state Gaussian HMM capture in equity returns?...
  output:      A 5-state HMM captures distinct market regimes: Bull, Neutral, Bear, S...

Row 5:
  instruction: How do you size a dispersion trade on BankNifty?...
  output:      Dispersion sizing uses vega-neutral weights: long constituent vol, sho...


---
## Bonus — Grok (xAI) & Gemini 

**llmreplay works across providers.** Here's how the same record/replay loop looks with Grok and Gemini — no real API keys needed, we simulate the response the same way as above.


In [94]:
# ── Grok (xAI) — OpenAI-compatible, detected via base_url ───────────────────
# In real usage:
#   import openai
#   client = openai.OpenAI(api_key=os.environ["XAI_API_KEY"], base_url="https://api.x.ai/v1")
#   with record("grok_run"):
#       response = client.chat.completions.create(model="grok-3", messages=[...])

GROK_RUN = "grok_demo"

with record(GROK_RUN, seed=42, base_dir=REPLAY_DIR, overwrite=True) as session:
    # Simulate a grok-3 response
    fake_llm_call(session, "What is the meaning of life?", model="grok-3", cost=0.006)

rs = replay(GROK_RUN, base_dir=REPLAY_DIR)
print(f"Grok run recorded: {rs.total_steps()} steps, cost ${rs.total_cost():.4f}")
for ev in rs.events():
    if ev.kind.value == "llm_request":
        print(f"  provider=grok  model={ev.payload.get('model')}  prompt={ev.payload.get('messages',[{}])[0].get('content','')[:60]}")


Grok run recorded: 4 steps, cost $0.0060
  provider=grok  model=grok-3  prompt=What is the meaning of life?


---
## Summary

| Use Case | What you got |
|---|---|
| 1️⃣ Production failure | Exact step + event that caused the crash, for free |
| 2️⃣ Cost audit | Per-step USD breakdown + ASCII heatmap |
| 3️⃣ Tool mocking | Replay with no live API calls, asserts exhaustion |
| 4️⃣ Regression testing | Rich pass/fail table, catches regressions before prod |
| 5️⃣ Fine-tuning export | JSONL + Alpaca dataset from your own runs |
| 🌐 Multi-provider | Grok + Gemini record/replay, same API, same cost tracking |

All five use cases run **offline**, cost **$0**, and produce **bitwise-identical** replays.
